# 퍼셉트론(Perceptrons, 인공뉴런 하나)
- 1943년 McCulloh & Pitts
    - 입력을 받아서 임계값을 넘으면 발화(1) 아니면 0인 논리 소자 모델링
- 1958년 Frank Rosenblatt
    - 가중치를 데이터로부터 학습하는 퍼셉트론 제안

## 구조
입력벡터 $x=(x_1,x_2,x_3,...,x_n)$이 들어오면
$$z = w_1x_1 + w_2x_2 + \dots + w_nx_n + b = \mathbf{w}^\top\mathbf{x} + b$$
$$\hat{y} = \text{step}(z) = \begin{cases} 1 & z \ge 0 \\ 0 & z < 0 \end{cases}$$

## 학습 규칙
틀릴때마다 다음처럼 갱신
$$\mathbf{w} \leftarrow \mathbf{w} + \eta\,(y - \hat{y})\,\mathbf{x}, \qquad b \leftarrow b + \eta\,(y - \hat{y})$$

#### 퍼셉트론 수렴정리
데이터가 선형분리 가능하면, 유한번 갱신안에 반드시 수렴한다.

## 한계
#### XOR 문제(Perceptrons, Minsky&Papert, 1969)
|x₁|x₂|AND|OR|XOR|
|:-|:-|:-|:-|:-|
|0|0|0|0|0|
|0|1|0|1|1|
|1|0|0|1|1|
|1|1|1|1|0|

AND와 OR은 직선 하나로 나눌 수 있지만, XOR은 직선 하나로 나눌 수 없습니다.

In [4]:
# 파이토치 설치
!uv add torch torchvision torchaudio

Resolved 146 packages in 356ms                                       
⠙ Preparing packages... (0/2)                                                   
⠙ Preparing packages... (0/2)-------------------     0 B/7.23 MiB            
⠙ Preparing packages... (0/2)------------------- 16.00 KiB/7.23 MiB          
⠙ Preparing packages... (0/2)------------------- 32.00 KiB/7.23 MiB          
⠙ Preparing packages... (0/2)------------------- 48.00 KiB/7.23 MiB          
⠙ Preparing packages... (0/2)------------------- 64.00 KiB/7.23 MiB          
⠙ Preparing packages... (0/2)------------------- 80.00 KiB/7.23 MiB          
⠙ Preparing packages... (0/2)------------------- 96.00 KiB/7.23 MiB          
⠙ Preparing packages... (0/2)------------------- 112.00 KiB/7.23 MiB         
⠙ Preparing packages... (0/2)------------------- 128.00 KiB/7.23 MiB         
⠙ Preparing packages... (0/2)------------------- 144.00 KiB/7.23 MiB         
⠙ Preparing packages... (0/2)------------------- 160.00 KiB/7.23 MiB 

In [ ]:
# 파이토치 설치 확인
import torch

print(torch.__version__)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0)) # GPU 이름
    print(torch.cuda.device_count()) # GPU 개수
    
    
# 애플 실리콘(M1/M2/M3) Mac은 MPS 백엔드 확인
print(torch.backends.mps.is_available())

2.15.0+cu132
True
NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
2
False


In [8]:
# 장치 선택 관용구
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 장치: {device}")

사용 장치: cuda


# 퍼셉트론 예제
PyTorch에서는 퍼셉트론(인공 뉴런 하나)에 대한 클래스나 함수를 지원하지 않고, 최소 선형 완전 연결층부터 지원하므로
PyTorch의 텐서를 입력으로 받는 예제 퍼셉트론을 구성한다.

퍼셉트론 구성 후, XOR문제에 대해 해결이 안되는지 검증한다.

## 구조
입력벡터 $x=(x_1,x_2,x_3,...,x_n)$이 들어오면
$$z = w_1x_1 + w_2x_2 + \dots + w_nx_n + b = \mathbf{w}^\top\mathbf{x} + b$$
$$\hat{y} = \text{step}(z) = \begin{cases} 1 & z \ge 0 \\ 0 & z < 0 \end{cases}$$

## 학습 규칙
틀릴때마다 다음처럼 갱신
$$\mathbf{w} \leftarrow \mathbf{w} + \eta\,(y - \hat{y})\,\mathbf{x}, \qquad b \leftarrow b + \eta\,(y - \hat{y})$$

In [35]:
class Perceptrons:
    def __init__(self, input_dim): # 퍼셉트론 생성 시에는 입력 크기를 지정한다.
        self.input_dim = input_dim
        self.weight_tensor = torch.randn(input_dim) # 입력 크기 만큼의 가중치 1차원 배열 생성 (정규분포 기반 난수 생성)
        self.bias = torch.randn(input_dim) # 입력 크기 만큼의 편향 배열 생성
        self.stop_flag = False
    def step(self, z): # Step 함수 정의
        return (z >= 0).float()
        
    def forward(self,/, x, y, lr): # 1회 학습
        z = self.weight_tensor * x.t() + self.bias
        y_hat = self.step(z)
        # 불리언 텐서의 .all 메서드는 모두 True이면 True를 반환한다.
        # 반대로 .any 메서드는 하나라도 True이면 True를 반환한다.
        if ((y == y_hat).all()): # 예측 결과가 같으면 학습 중단
            self.stop_flag = True
            return
        # 가중치, 편향 업데이트
        self.weight_tensor = self.weight_tensor + (lr * (y - y_hat) * x)
        self.bias = self.bias + (lr * (y - y_hat))
        
    def train(self, x, y, /, step=10, lr=0.1): # 스텝 만큼의 학습 호출
        for st in range(step):
            self.forward(x=x, y=y, lr=lr)
            if self.stop_flag: return True
            print(f"step {st+1}: weights={self.weight_tensor}, bias={self.bias}")
        
        return False
            
    def test(self, x): # x를 받아서 y 출력해보기
        z = self.weight_tensor * x.t() + self.bias
        y_hat = self.step(z)
        return y_hat

In [38]:
# 예제 데이터 준비
x = torch.tensor([1, 3, 5, 6])
y = torch.tensor([1, 1, 0, 1])

perceptrons = Perceptrons(4)
epoch = 3

for i in range(epoch):
    print(f"Epoch {i+1} Start")
    if (perceptrons.train(x, y)):
        print(f"Epoch {i+1} End")
        break
    print(f"Epoch {i+1} End")
    

print(f"Final Weight: {perceptrons.weight_tensor}") # 최종 가중치
print(f"Test Result: y_hat: {perceptrons.test(x)}, y: {y}") # 예상 값과 동일한지 점검.

Epoch 1 Start
step 1: weights=tensor([1.3759, 1.8939, 0.6133, 0.5568]), bias=tensor([ 0.2793, -1.3558, -0.0541, -0.0578])
step 2: weights=tensor([1.3759, 1.8939, 0.1133, 0.5568]), bias=tensor([ 0.2793, -1.3558, -0.1541, -0.0578])
step 3: weights=tensor([ 1.3759,  1.8939, -0.3867,  0.5568]), bias=tensor([ 0.2793, -1.3558, -0.2541, -0.0578])
Epoch 1 End
Final Weight: tensor([ 1.3759,  1.8939, -0.3867,  0.5568])
Test Result: y_hat: tensor([1., 1., 0., 1.]), y: tensor([1, 1, 0, 1])
